In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
train_path = "../artifacts/train.csv"
val_path = "../artifacts/validation.csv"
test_path = "../artifacts/test.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
def create_features(df):
    df = df.copy()

    # Convert dates
    df["purchase_dt"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["estimated_delivery_dt"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

    # Estimated delivery window
    df["estimated_delivery_days"] = (
        df["estimated_delivery_dt"]
        - df["purchase_dt"]
    ).dt.total_seconds() / (24 * 3600)

    # Time-based features
    df["purchase_year"] = df["purchase_dt"].dt.year
    df["purchase_month"] = df["purchase_dt"].dt.month
    df["purchase_dayofweek"] = df["purchase_dt"].dt.dayofweek
    df["purchase_hour"] = df["purchase_dt"].dt.hour

    return df

In [ ]:
train_df = create_features(train_df)
val_df = create_features(val_df)
test_df = create_features(test_df)

print("Feature engineering completed successfully.")

In [ ]:
engineered_features = [
    "purchase_dt",
    "estimated_delivery_dt",
    "estimated_delivery_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour"
]

print("=== ENGINEERED FEATURES ===")

print(train_df[engineered_features].head())

print("\nEstimated delivery days:")
print(
    train_df["estimated_delivery_days"].describe()
)

In [ ]:
feature_cols = [
    "estimated_delivery_days",
    "item_count",
    "total_items_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_count",
    "total_payment_value",
    "payment_types",
    "max_installments",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour"
]

print("Number of features:", len(feature_cols))

print("\nFinal features:")
for feature in feature_cols:
    print("-", feature)

In [ ]:
X_train = train_df[feature_cols].copy()
y_train = train_df["late"].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df["late"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["late"].copy()

print("Train:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTest:")
print("X:", X_test.shape)
print("y:", y_test.shape)

In [ ]:
numeric_features = [
    "estimated_delivery_days",
    "item_count",
    "total_items_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_count",
    "total_payment_value",
    "payment_types",
    "max_installments",
    "customer_zip_code_prefix",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour"
]

categorical_features = [
    "customer_city",
    "customer_state"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numeric pipeline created.")

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5
        )
    )
])

print("Categorical pipeline created.")

In [ ]:
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

print("Preprocessor created successfully.")

In [ ]:
print("Fitting preprocessor on TRAIN only...")

preprocessor.fit(X_train)

print("Preprocessor fitted successfully.")

In [ ]:
X_train_transformed = preprocessor.transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

print("Transformation completed.")

print("\nTransformed shapes:")
print("Train:", X_train_transformed.shape)
print("Validation:", X_val_transformed.shape)
print("Test:", X_test_transformed.shape)

In [ ]:
print("=== MISSING VALUES BEFORE TRANSFORMATION ===")

print("Train missing:", X_train.isna().sum().sum())
print("Validation missing:", X_val.isna().sum().sum())
print("Test missing:", X_test.isna().sum().sum())

In [ ]:
print("=== TARGET CHECK ===")

print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

In [ ]:
forbidden_features = [
    "late",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "review_score"
]

leakage_found = [
    col for col in feature_cols
    if col in forbidden_features
]

print("=== LEAKAGE CHECK ===")

if len(leakage_found) == 0:
    print("No leakage features found.")
else:
    print("Potential leakage:")
    print(leakage_found)

In [ ]:
import json

feature_artifact = {
    "features": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

with open(
    "../artifacts/feature_config.json",
    "w"
) as f:
    json.dump(
        feature_artifact,
        f,
        indent=4
    )

print("Feature configuration saved.")

In [ ]:
import joblib

joblib.dump(
    preprocessor,
    "../artifacts/preprocessor.joblib"
)

print("Preprocessor saved successfully.")

In [ ]:
train_features = X_train.copy()
train_features["late"] = y_train.values

val_features = X_val.copy()
val_features["late"] = y_val.values

test_features = X_test.copy()
test_features["late"] = y_test.values

train_features.to_csv(
    "../artifacts/train_features.csv",
    index=False
)

val_features.to_csv(
    "../artifacts/validation_features.csv",
    index=False
)

test_features.to_csv(
    "../artifacts/test_features.csv",
    index=False
)

print("Feature tables saved successfully.")

In [ ]:
print("======================================")
print("NOTEBOOK 5 — FINAL CHECK")
print("======================================")

print("\nFeature count:")
print(len(feature_cols))

print("\nTrain:")
print(X_train.shape)

print("\nValidation:")
print(X_val.shape)

print("\nTest:")
print(X_test.shape)

print("\nArtifacts created:")
print("- feature_config.json")
print("- preprocessor.joblib")
print("- train_features.csv")
print("- validation_features.csv")
print("- test_features.csv")

print("\nNotebook 5 completed successfully.")